# 00 - Environment Understanding

Goal: understand what SoccerTwos gives the agent before training anything. This notebook checks imports, compares the training and live-match action spaces, runs a random rollout, and inspects the `info` fields used by reward shaping.

## Setup

Run this from the existing `soccertwos` environment. No environment creation happens here.

In [8]:
from pathlib import Path
import sys

def _add_project_root_to_path():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "soccer-twos-starter"):
            if (candidate / "soccer_twos_project" / "notebook_tools.py").exists():
                if str(candidate) not in sys.path:
                    sys.path.insert(0, str(candidate))
                return candidate
    raise FileNotFoundError("Could not find the soccer-twos-starter project root.")

_add_project_root_to_path()

import importlib
import soccer_twos_project.notebook_tools as notebook_tools
importlib.reload(notebook_tools)
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

Project root: /Users/vedaangchopra/all_data/Georgia Tech/Course Content/CS 8803- DRL/project/soccer-twos-starter
Artifact root: /Users/vedaangchopra/all_data/Georgia Tech/Course Content/CS 8803- DRL/project/soccer-twos-starter/artifacts/cs8803_soccer_twos
Python: /opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/bin/python
soccer_twos: /opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/lib/python3.8/site-packages/soccer_twos/__init__.py
ray: 1.13.0
torch: 1.13.1
python: /opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/bin/python
{
  "cpu_count": 16,
  "gpu_name": "",
  "mlx_available": false,
  "ram_gb": 64.0,
  "torch_cuda_available": false,
  "torch_mps_available": true
}


## Environment Gate

Expected facts: single-player training uses a flat `(336,)` observation and `Discrete(27)` action space; live evaluation uses a dict for four players and `MultiDiscrete([3, 3, 3])` actions.

In [9]:
run_environment_gate(steps=10)

{
  "gym": "0.19.0",
  "numpy": "1.23.5",
  "python": "/opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/bin/python",
  "ray": "1.13.0",
  "soccer_twos": "/opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/lib/python3.8/site-packages/soccer_twos/__init__.py",
  "torch": "1.13.1"
}
Using Unity base_port: 50039


I0000 00:00:1776831083.365075  142363 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


...

[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Gate reset observation shape: (336,)
First-step info: {'player_info': {'position': [-9.025915145874023, 1.0617339611053467], 'rotation_y': 87.72977447509766, 'velocity': [0.08774314820766449, -2.213289737701416]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
Environment gate passed. steps=10 total_reward=0.000


## Random Rollout Logs

Rewards are usually zero because goals are rare. The `info` field exposes player and ball state, which is useful for training-only reward shaping.

In [10]:
run_random_debug_episode(max_steps=10, show_info=True)

Using Unity base_port: 50042


I0000 00:00:1776831089.788682  142363 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Observation type: <class 'numpy.ndarray'>
Observation shape: (336,)
Action space: Discrete(27)
step=01 action=10 reward=0.0 done=False info={'player_info': {'position': [-8.530884742736816, 1.2444617748260498], 'rotation_y': 77.7297592163086, 'velocity': [7.988476276397705, 0.9225671291351318]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
step=02 action=26 reward=0.0 done=False info={'player_info': {'position': [-7.907595157623291, 1.4700276851654053], 'rotation_y': 87.72978210449219, 'velocity': [5.248717308044434, 3.0490427017211914]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
step=03 action=9 reward=0.0 done=False info={'player_info': {'position': [-6.966447830200195, 1.7392423152923584], 'rotation_y': 87.72978210449219, 'velocity': [11.909119606018066, 2.4877548217773438]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
step=04 action=23 rew

## Reward Shaping Signal

The shaped agent keeps the environment reward, then adds a small clipped bonus for moving toward the ball and moving the ball toward the opponent goal. This code path is used only during training; exported agents do not depend on the wrapper.

In [11]:
import soccer_twos
from soccer_twos import EnvType
from soccer_twos_project.envs import RewardShapingWrapper

env = RewardShapingWrapper(
    make_soccer_env(render=True, variation=EnvType.team_vs_policy, flatten_branched=True, single_player=True),
    player_to_ball_weight=0.01,
    ball_to_goal_weight=0.02,
    clip=0.05,
)
try:
    obs = env.reset()
    for step in range(5):
        obs, reward, done, info = env.step(env.action_space.sample())
        print(f"step={step + 1} shaped_reward={reward:.4f} info={compact_info(info)}")
finally:
    env.close()

Using Unity base_port: 50043


I0000 00:00:1776831093.209911  142363 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


step=1 shaped_reward=0.0000 info={'player_info': {'position': [-9.03687858581543, 1.338266134262085], 'rotation_y': 87.72977447509766, 'velocity': [-0.08774314820766449, 2.213289737701416]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
step=2 shaped_reward=0.0000 info={'player_info': {'position': [-9.043486595153809, 1.5049329996109009], 'rotation_y': 87.72977447509766, 'velocity': [-0.05305497720837593, 1.3382928371429443]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
step=3 shaped_reward=0.0000 info={'player_info': {'position': [-9.047178268432617, 1.4600732326507568], 'rotation_y': 97.72977447509766, 'velocity': [-0.09670231491327286, -1.4063316583633423]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
step=4 shaped_reward=0.0000 info={'player_info': {'position': [-9.081583023071289, 1.207076072692871], 'rotation_y': 107.72978973388672, 'veloci